# 信用卡数字识别

In [15]:
import cv2
import numpy as np
import utils
from CV.utils import cv_show

In [16]:
# 信用卡类型
card_types = {
    "3": "American Express",
    "4": "Visa",
    "5": "Mastercard",
    "6": "Discover Card"
}

In [17]:
image = cv2.imread("./images/credit/ocr_a_reference.png")
template = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
_, template = cv2.threshold(template, 10, 255, cv2.THRESH_BINARY_INV)
contours, _ = cv2.findContours(template.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
image = cv2.drawContours(image, contours, -1, (0, 0, 255), 2)
utils.cv_show("template", image)

In [18]:
contours, _ = utils.sort_contours(contours, method="left-to-right")
# 每个数字对应一个轮廓
numbers = {}
for i, c in enumerate(contours):
    x, y, w, h = cv2.boundingRect(c)
    roi = template[y:y+h, x:x+w]
    roi = cv2.resize(roi, (57, 88))
    numbers[i] = roi

In [19]:
rect_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (9, 3))
sq_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))
image = cv2.imread("./images/credit/credit_card_02.png")
image = utils.resize(image, width=300)
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
cv_show("gray", gray)

In [20]:
# tophat 操作，突出更明亮的区域
tophat = cv2.morphologyEx(gray, cv2.MORPH_TOPHAT, rect_kernel)
cv_show("tophat", tophat)

In [21]:
grad_x = cv2.Sobel(tophat, cv2.CV_32F, 1, 0, ksize=-1)
grad_x = np.absolute(grad_x)
(minVal, maxVal) = (np.min(grad_x), np.max(grad_x))
grad_x = 255 * ((grad_x - minVal) / (maxVal - minVal))
grad_x = grad_x.astype("uint8")

# grad_y = cv2.Sobel(tophat, cv2.CV_32F, 0, 1, ksize=-1)
# grad_y = np.absolute(grad_y)
# (minVal, maxVal) = (np.min(grad_y), np.max(grad_y))
# grad_y = 255 * ((grad_y - minVal) / (maxVal - minVal))
# grad_y = grad_y.astype("uint8")

grad = grad_x.copy()
cv_show("grad_x", grad)

In [22]:
# 通过闭操作将数字连在一起
grad = cv2.morphologyEx(grad, cv2.MORPH_CLOSE, rect_kernel)
cv_show("grad_close", grad)

In [23]:
thresh = cv2.threshold(grad, 0, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU)[1]
cv_show("thresh", thresh)

In [24]:
# 再进行闭操作
thresh = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, sq_kernel)
cv_show("thresh_close", thresh)

In [25]:
# 计算轮廓
contours, _ = cv2.findContours(thresh.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
draw_image = image.copy()
cv2.drawContours(draw_image, contours, -1, (0, 0, 255), 2)
cv_show("draw_image", draw_image)

In [26]:
locations = []
for i, c in enumerate(contours):
    x, y, w, h = cv2.boundingRect(c)
    r = w / h
    if 2.5 < r < 4.0 and 40 < w < 55 and 10 < h < 20:
        locations.append((x, y, w, h))
locations = sorted(locations, key=lambda x: x[0])

In [27]:
output = []
for i, (x, y, w, h) in enumerate(locations):
    temp = []
    group = gray[y-5:y+h+5, x-5:x+w+5]
    group = cv2.threshold(group, 0, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU)[1]
    group_contours, _ = cv2.findContours(group.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    group_contours, _ = utils.sort_contours(group_contours, method="left-to-right")
    for c in group_contours:
        gx, gy, gw, gh = cv2.boundingRect(c)
        roi = group[gy:gy+gh, gx:gx+gw]
        roi = cv2.resize(roi, (57, 88))
        
        scores = []
        for num, num_roi in numbers.items():
            result = cv2.matchTemplate(roi, num_roi, cv2.TM_CCOEFF_NORMED)
            _, score, _, _ = cv2.minMaxLoc(result)
            scores.append(score)
        temp.append(str(np.argmax(scores)))
    cv2.rectangle(image, (x-5, y-5), (x+w+5, y+h+5), (0, 0, 255), 1)
    image = cv2.putText(image, "".join(temp), (x, y - 15), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 0, 255), 2)
    output.extend(temp)

In [28]:
print(f"信用卡类型: {card_types[output[0]]}")
print(f"卡号: {''.join(output)}")
cv_show("image", image)

信用卡类型: Visa
卡号: 4020340002345678
